# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore the [FAIR²](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. We will load the Croissant schema, examine the dataset structure, extract records, perform basic analysis, and visualize key data relationships.

### Dataset Source

The dataset is published according to the Croissant standard and accessible via the following schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets and their field structure. 
Each entity is referenced by its Croissant `@id`.

We will enumerate all the record sets, list their `@id`, and display the available fields (with their `@id`s and names) for each. This ensures you can identify the correct identifiers for subsequent extraction and analysis.

In [ ]:
# List all record sets with their @id and associated fields
record_sets = list(dataset.record_sets)

if not record_sets:
    print("No record sets found in the dataset schema! Please check the schema structure.")
else:
    for rset in record_sets:
        print(f"Record Set Name: {rset.name}")
        print(f"  @id: {rset.id}")
        print("  Fields:")
        for field in rset.fields:
            print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
        print()

## 3. Data Extraction
Load data from selected record sets into DataFrames for analysis. Below, we demonstrate extraction for all record sets identified above. Make sure you use the correct `@id` identifiers; this ensures your code will keep working if the dataset evolves.

In [ ]:
# Prepare for extraction
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for recset_id in record_set_ids:
    records_iter = dataset.records(record_set=recset_id)
    df = pd.DataFrame(records_iter)
    dataframes[recset_id] = df
    print(f"Loaded {len(df)} records from record set {recset_id}")

# Choose the first record set for further demonstration
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    df = dataframes[main_record_set_id]
    print(f"\nRecord set columns (@id): {df.columns.tolist()}")
    display(df.head())
else:
    print('No record sets available for extraction.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes. This section demonstrates removing outliers, normalizing fields, and basic grouping/aggregation. Ensure all field references use their `@id` as column names.

In [ ]:
import numpy as np

# Preview available numeric fields for selection
numeric_columns = []
for col in df.columns:
    # Try to convert column to numeric; if succeeds, treat as numeric field
    try:
        pd.to_numeric(df[col].dropna().iloc[:5])
        numeric_columns.append(col)
    except Exception:
        pass

print("Numeric fields detected (by @id):", numeric_columns)

# For demonstration, select the first numeric field if available
if numeric_columns:
    numeric_field_id = numeric_columns[0]
    print(f"\nAnalyzing numeric field: {numeric_field_id}")

    # Drop NaN and convert to numeric for analysis
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    threshold = df[numeric_field_id].quantile(0.75)  # Top quartile as filter example
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df[[numeric_field_id]].head())

    # Normalize
    norm_col = numeric_field_id + '_normalized'
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nFirst 5 normalized values for {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Attempt grouping by a non-numeric field
    group_field = None
    for col in df.columns:
        if col != numeric_field_id and df[col].dtype == 'object':
            group_field = col
            break

    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean {numeric_field_id} by {group_field} (top 5 groups):")
        display(grouped_df.head())
else:
    print("No numeric fields found for EDA.")

## 5. Visualization
Visualize the distribution of a numeric field and its relationships to categorical groupings using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot if a group field is available
    if group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, you learned to load and explore the FAIR² mass spectrometry dataset using the Croissant standard with `mlcroissant`. You reviewed the dataset structure by record set and field `@id`, extracted records, performed basic filtering and normalization, and created visualizations of numeric columns. For advanced analyses, further leverage the semantic metadata for robust, reproducible exploration!

Refer to the dataset's Croissant schema for in-depth details on field semantics, provenance, and licensing at: https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json